# Informe Final Integral: Análisis de la Industria de Videojuegos con Datos de IGDB

## Resumen Ejecutivo
Este documento consolida el ciclo de vida completo del proyecto de Ciencia de Datos, desde la ingestión de datos crudos hasta el modelado predictivo. El objetivo principal es transformar datos técnicos en decisiones estratégicas para desarrolladores de videojuegos independientes ("Indies"), identificando los factores clave que separan a un juego exitoso del resto.

---

## 1. Fase de Extracción de Datos (ETL - Extraction)
**Notebook de referencia:** `igdb_api_request.ipynb`

En esta etapa, definimos la estrategia para obtener la materia prima del proyecto.

### 1.1. Selección de la Fuente de Datos
Optamos por utilizar la **API oficial de IGDB** como nuestra fuente de datos **Secundaria y Externa**.
* **Justificación Técnica:** A diferencia del *Web Scraping*, que extrae datos no estructurados (HTML) y es vulnerable a cambios en el diseño web, la API nos proporciona datos **semi-estructurados (JSON)**. Esto garantiza una integridad de datos superior y relaciones pre-definidas entre entidades.
* **Justificación Ética:** El uso de una API documentada respeta los términos de servicio del proveedor y evita la sobrecarga innecesaria de sus servidores, alineándose con las buenas prácticas de la ingeniería de datos.

### 1.2. Metodología de Extracción
Implementamos un proceso de descarga iterativa (paginación).
* **Desafío:** La base de datos contiene cientos de miles de registros. Una sola petición HTTP habría excedido los límites de tiempo y memoria.
* **Solución:** Se diseñó un bucle que solicita lotes (`limit=500`) utilizando un puntero de desplazamiento (`offset`).
* **Control de Flujo:** Se implementó un retardo artificial (`time.sleep`) entre peticiones.
    * *Por qué:* Para cumplir estrictamente con los límites de tasa (*Rate Limits*) de la API y evitar bloqueos de IP, garantizando la reproducibilidad del proceso.

---

## 2. Fase de Limpieza y Transformación (Data Wrangling)
**Notebook de referencia:** `00_limpieza_raw_g.ipynb` y `01_limpieza_auxiliares.ipynb`

Aquí abordamos la "T" de ETL, transformando datos crudos en un formato analítico confiable.

### 2.1. Manejo de Tipos de Datos Complejos
* **Problema:** Al exportar JSON a CSV, las estructuras de listas (ej: `[1, 2, 3]`) se convirtieron en cadenas de texto (`string`), perdiendo su funcionalidad.
* **Solución:** Utilizamos `literal_eval` de la librería `ast`.
* **Justificación:** Esto permitió recuperar la estructura de lista nativa de Python de manera segura (a diferencia de `eval` que ejecuta código arbitrario), habilitando la posterior expansión de géneros y plataformas.

### 2.2. Definición de la Ventana Temporal y Calidad
* **Decisión:** Se filtraron juegos lanzados exclusivamente entre **2000 y 2025**.
* **Justificación:** El análisis de calidad del dato (*Data Profiling*) reveló que los registros anteriores al año 2000 presentaban un alto porcentaje de valores nulos en métricas críticas (ratings de usuarios). Incluirlos habría introducido ruido y sesgo histórico no representativo del mercado actual.

### 2.3. Lógica de Negocio para Duplicados
* **Hallazgo:** Un mismo título aparece múltiples veces debido a re-lanzamientos, ediciones "GOTY" o ports regionales.
* **Tratamiento:** En lugar de una eliminación aleatoria, aplicamos una lógica de negocio: **Mantener la versión con mayor `total_rating_count`**.
* **Por qué:** Asumimos que la versión con más interacciones de usuarios es la "principal" o la que mejor refleja la recepción del mercado, preservando así la información de mayor valor.

---

## 3. Fase de Análisis Exploratorio de Datos (EDA)
**Notebook de referencia:** `02_eda_igdb.ipynb`

En esta fase, buscamos comprender las distribuciones y validar hipótesis antes del modelado.

### 3.1. Justificación Metodológica: Datos Faltantes (MNAR)
Identificamos que la variable `total_rating` tenía un alto porcentaje de nulos.
* **Conclusión Teórica:** Estos datos no faltan al azar (MCAR). Se trata de un mecanismo **MNAR (Missing Not At Random)**: los juegos no tienen rating porque son impopulares o desconocidos.
* **Decisión:** Para el análisis de "Éxito", se trabajó solo con el subconjunto de datos que posee métricas, asumiendo que la falta de datos es, en sí misma, un indicador de falta de éxito comercial.

### 3.2. Ingeniería de la Variable Objetivo ($Y$)
Para habilitar el Aprendizaje Supervisado, necesitábamos una definición concreta de "Éxito".
* **Definición:** `Exitoso = (Rating >= 75) AND (Votos >= 10)`.
* **Justificación:**
    * *Rating:* Filtra la calidad (75/100 es un estándar de "buen juego").
    * *Votos:* Filtra la relevancia estadística. Un juego con un solo voto de 100/100 no es un éxito de mercado; es un dato atípico (*outlier*) o sesgado.

### 3.3. Conclusiones Visuales y Respuestas a Preguntas de Negocio
A continuación, presentamos los hallazgos que responden a las preguntas planteadas al inicio del proyecto.

#### A. ¿Qué géneros dominan el mercado? (Cantidad vs. Calidad)
Al cruzar el año de lanzamiento con el género y la tasa de éxito, obtenemos el siguiente mapa de calor:

![Mapa de Calor: Éxito por Género y Año](img/mapa_calorr.png)

* **Volumen vs. Éxito:** Aunque el género **"Indie"** satura el mercado en cantidad, el mapa revela que su tasa de éxito (colores claros) es baja comparada con otros.
* **Nichos Fuertes:** Géneros de mayor complejidad como **RPG (Role-playing)** y **Strategy** muestran colores más intensos consistentemente. Esto sugiere que, aunque hay menos juegos, tienen bases de jugadores más fieles y críticas más sólidas.

#### B. La Brecha AAA vs. Indie
Para entender las diferencias estructurales, comparamos las distribuciones de ratings y la cantidad de reseñas (visibilidad) según el tipo de estudio:

![Gráfico de Barras: Comparación AAA vs No AAA](img/aaa_noaaa.png)

* **Probabilidad de Éxito:** Existe un abismo estadístico. Un juego AAA tiene un **36.6%** de probabilidad de éxito, mientras que un Indie sin publisher cae al **2.7%**.
* **El Factor Visibilidad:** Como se observa en el gráfico de la derecha (escala logarítmica), la mediana de reseñas para un juego AAA es inmensamente superior.
* **Conclusión:** El músculo de marketing es determinante. Sin visibilidad (reseñas), no hay éxito en nuestro modelo, incluso si el juego tiene buena nota técnica (gráfico izquierdo).

#### C. Estilos de Juego y Comunidad

* Observamos que los juegos con ciclos de vida cortos (Arcade/Casual) tienen dificultades para acumular las 10 reseñas necesarias.
* Los juegos profundos fomentan comunidades que discuten y reseñan, otorgando **visibilidad orgánica** a largo plazo.

#### D. Plataformas: ¿Dónde debe lanzar un Indie?
Al filtrar exclusivamente por juegos *No AAA*, analizamos qué plataformas ofrecen mejor tasa de éxito:

![Gráfico de Barras: Éxito por Plataforma](img/exito_plat.png)

* **PC y Consolas:** Los datos sugieren que estas plataformas ofrecen mayor probabilidad de éxito que los móviles.
* **Por qué:** Las tiendas de aplicaciones móviles sufren graves problemas de descubribilidad. En PC/Consola, el público está más predispuesto a interactuar y calificar el producto.

#### E. El Mito de la Estacionalidad (Ventanas de Lanzamiento)
Analizamos la proporción de éxito según el mes de lanzamiento para validar si existen "mejores momentos" para publicar:

![Gráfico de Barras: Éxito por Mes](img/exito_mes.png)

* **Enero (8.4% éxito):** El peor mes, posiblemente debido a la "resaca de gastos" post-navidad.
* **Septiembre/Octubre (~13% éxito):** Los picos más altos de éxito para indies.
* **Estrategia:** Existe una ventana de atención clara en el tercer trimestre (Q3), coincidiendo con el regreso a la rutina, que los desarrolladores independientes están aprovechando mejor que el inicio del año.

---

## 4. Fase de Modelado Predictivo (Machine Learning)
**Notebook de referencia:** `03_modelo.ipynb`

Finalmente, construimos modelos para predecir la probabilidad de éxito $P(Y=1|X)$.

### 4.1. Ingeniería de Características (Feature Engineering)
* **One-Hot Encoding:** Transformamos variables categóricas (Géneros, Plataformas) en vectores binarios.
* **Reducción de Dimensionalidad:** Seleccionamos solo los **Top 10 géneros** más frecuentes para evitar la dispersión de datos (*sparsity*), lo cual podría degradar el rendimiento del modelo.

### 4.2. Manejo del Desbalance de Clases
* **Diagnóstico:** Solo el ~13% de los juegos cumplían los criterios de éxito. Un modelo ingenuo podría predecir siempre "No Exitoso" y tener un 87% de exactitud (*Accuracy*), pero sería inútil.
* **Solución:** Utilizamos el hiperparámetro `class_weight='balanced'` en los algoritmos y estratificamos la división de datos (`stratify=y`) para asegurar representatividad en los sets de entrenamiento y prueba.

### 4.3. Resultados de los Modelos
* **Regresión Logística:**
    * *Rol:* Modelo base (*Baseline*) interpretable.
    * *Hallazgo:* Los coeficientes ($\beta$) confirmaron que ser "Sin Publisher" penaliza gravemente la probabilidad de éxito, mientras que lanzar en "Consola" la aumenta.
* **Random Forest:**
    * *Rol:* Modelo no lineal robusto.
    * *Hallazgo:* Superó ligeramente en métricas de clasificación (ROC-AUC ~0.78) al capturar interacciones complejas entre variables (ej: un RPG en PC funciona diferente que un RPG en Móvil).

---

# 5. Conclusiones Generales y Recomendaciones Estratégicas

Basándonos en la evidencia empírica recolectada y modelada a lo largo de todo el proyecto, presentamos las siguientes recomendaciones para un estudio de videojuegos independiente:

### A. El Dilema del Publisher (Visibilidad vs. Autonomía)
Los datos son concluyentes: la variable más predictiva del fracaso es no tener Publisher.
* **Recomendación:** La "calidad" del juego no es suficiente. Se recomienda buscar activamente un socio de publicación o invertir agresivamente en marketing. La visibilidad orgánica es prácticamente inexistente en el mercado actual.

### B. Selección de Nicho (Estrategia de Océano Azul)
El mercado de juegos etiquetados genéricamente como "Indie" o "Arcade" es un "Océano Rojo" (alta competencia, bajo retorno).
* **Recomendación:** Desarrollar para nichos de alta fidelidad. **Estrategia, Simuladores y RPGs** tienen comunidades más activas que generan las reseñas necesarias para cumplir los criterios de éxito del algoritmo de las tiendas.

### C. Plataformas y Ventanas de Lanzamiento
* **Plataforma:** Evitar el lanzamiento exclusivo en móviles si se busca prestigio crítico. **PC y Consolas** ofrecen probabilidades de éxito significativamente mayores.
* **Fecha:** Planificar el lanzamiento para el **Tercer Trimestre (Q3)**, específicamente Septiembre/Octubre. Evitar a toda costa el lanzamiento en Enero, donde la propensión del consumidor a interactuar con nuevos productos es la más baja del año.

---
*Este proyecto demuestra cómo la aplicación rigurosa de técnicas de Ciencia de Datos (ETL, EDA, ML) permite transformar la intuición en decisiones de negocio informadas y estadísticamente validadas.*